In [ ]:
import os
import torch
import asyncio
import logging
import polars as pl
import nest_asyncio
from dotenv import load_dotenv
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from datasets import Dataset
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from huggingface_hub import login
import uuid

In [ ]:
# Aplicar nest_asyncio para evitar erros de loop em Jupyter Notebook
nest_asyncio.apply()

# Configuração de logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Carregar variáveis de ambiente
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    login(token=hf_token)
else:
    raise ValueError("O token da Hugging Face (HF_TOKEN) não foi encontrado no arquivo .env")


In [ ]:
# Configurações globais
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Carregar modelo e tokenizer globalmente
logging.info(f"Carregando modelo: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")

# Carregar embeddings e ChromaDB
logging.info("Inicializando ChromaDB...")
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
vectorstore = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)


In [ ]:
def load_fine_tuning_dataset():
    """Carrega o dataset de fine-tuning a partir de um arquivo JSON."""
    logging.info("Carregando dataset de fine-tuning...")
    data = pl.read_ndjson("trn.json")
    return data.select(["title", "content"]).fill_null("")

In [ ]:
def load_train_test_datasets():
    """Carrega os datasets de treino e teste a partir de arquivos TXT."""
    logging.info("Carregando datasets de treino e teste...")
    dataset = {}
    for key, filename in zip(["train", "test"], ["filter_labels_train.txt", "filter_labels_test.txt"]):
        data = pl.read_csv(filename, separator="\n", has_header=False).to_pandas()
        dataset[key] = pl.DataFrame({"question": data[0::2].reset_index(drop=True),
                                     "answer": data[1::2].reset_index(drop=True)})
    return dataset


In [ ]:
async def persist_data_to_chromadb(dataset, batch_size=10000):
    """
    Persiste os dados no ChromaDB de forma assíncrona, utilizando inserção em lote.
    """
    texts = (dataset["title"] + " " + dataset["content"]).to_list()
    batch_size = min(batch_size, 500)  
    
    tasks = [
        vectorstore.aadd_documents([
            {"id": str(uuid.uuid4()), "text": text} for text in texts[i:i + batch_size]
        ])
        for i in range(0, len(texts), batch_size)
    ]
    
    await asyncio.gather(*tasks)
    logging.info("Dados persistidos com sucesso.")

In [ ]:
def retrieve_context(question):
    """Recupera contexto relevante do ChromaDB."""
    logging.info(f"Buscando contexto para a pergunta: {question}")
    docs = vectorstore.similarity_search(question, k=2)
    return " ".join([doc.page_content for doc in docs])


In [ ]:
def generate_response(question):
    """Gera uma resposta otimizada baseada na pergunta."""
    logging.info("Iniciando geração de resposta...")
    
    context = retrieve_context(question)
    prompt = f"Contexto: {context}\nPergunta: {question}\nResposta:"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.7, top_p=0.9)

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    logging.info("Resposta gerada com sucesso.")
    return response


In [ ]:
def fine_tune_model(dataset):
    """Realiza o fine-tuning do modelo Llama 3.1."""
    logging.info("Iniciando fine-tuning do modelo...")
    
    def tokenize_function(examples):
        return tokenizer(examples["content"], padding="max_length", truncation=True)

    train_dataset = Dataset.from_pandas(dataset.to_pandas())
    tokenized_datasets = train_dataset.map(tokenize_function, batched=True)

    training_args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=3,
        per_device_train_batch_size=2,  # Ajuste para evitar estouro de RAM
        logging_dir="./logs"
    )

    trainer = Trainer(model=model, args=training_args, train_dataset=tokenized_datasets)
    trainer.train()

    logging.info("Fine-tuning concluído.")
    return model


In [ ]:
# Teste de geração de resposta
pergunta = "Qual é o melhor fone de ouvido?"
resposta = generate_response(pergunta)
print(f"Resposta: {resposta}")

# Executar persistência de dados no Jupyter Notebook
await persist_data_to_chromadb(load_fine_tuning_dataset())